##### imports and reading in csv 

In [2]:
import pandas as pd
from datasets import load_dataset
ds = load_dataset("holodata/sensai")
flagged = ds['train'].to_pandas()


flagged.head(100)


Repo card metadata block was not found. Setting CardData to empty.


,body,label
0,Coordinatesw,hidden
1,hmmmmmmmmmmmmmmmm,hidden
2,you do know what this is tho,hidden
3,you do know what this is tho,hidden
4,you do know what this is tho,hidden
...,...,...
95,#AmeBeliever,hidden
96,#AmeBeliever,hidden
97,You can do it Amelia! �,hidden
98,no practice run,hidden


##### mapping data

In [3]:
# map 'hidden' and 'deleted' to 1 (delete), and 'nonflagged' to 0 (keep)
flagged['label'] = flagged['label'].map({
    'hidden': 1,
    'deleted': 1,
    'nonflagged': 0
})

##### data cleaning

In [4]:
print(f"{flagged['label'].value_counts()}")

original_size= flagged.shape[0]

flagged['body'] = flagged['body'].str.lower() # normalise the text
flagged = flagged.drop_duplicates(subset=['body']) # drop duplicates 
flagged['body'] = flagged['body'].str.strip() # remove unecessary whitespace 

flagged = flagged.dropna() # remove empty cells

new_size = flagged.shape[0]

print(f"{original_size - new_size} rows dropped")

print(f"{flagged['label'].value_counts()}")


label
1    3850036
0    3850036
Name: count, dtype: int64
4271945 rows dropped
label
0    1922024
1    1506103
Name: count, dtype: int64


##### splititng data (train-test)

In [5]:
from sklearn.model_selection import train_test_split

# define features (text) and targets (binary labels: 0 or 1)
text = flagged['body']
label = flagged['label']

# 80:20 train-test split
text_train, text_test, label_train, label_test = train_test_split(
    text, label, test_size=0.2, random_state=42, stratify=label)

##### training the model

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize vectorizer (limiting to top 10,000 features to save memory and speed up training)
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')

# Fit on training data and transform both sets
text_train_vec = vectorizer.fit_transform(text_train)
text_test_vec = vectorizer.transform(text_test)

from sklearn.linear_model import LogisticRegression

# Initialize and train the model
model = LogisticRegression(max_iter=1000)
model.fit(text_train_vec, label_train)

from sklearn.metrics import classification_report, confusion_matrix

# Make predictions on test set
label_pred = model.predict(text_test_vec)

# Display evaluation metrics
print("--- Classification Report ---")
print(classification_report(label_test, label_pred))

print("--- Confusion Matrix ---")
print(confusion_matrix(label_test, label_pred))

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.65      0.88      0.75    384405
           1       0.72      0.38      0.50    301221

    accuracy                           0.66    685626
   macro avg       0.68      0.63      0.62    685626
weighted avg       0.68      0.66      0.64    685626

--- Confusion Matrix ---
[[339612  44793]
 [185497 115724]]
